# Лабораторная работа «Свёрточные нейронные сети»
## Часть 1. Pet Faces

Решаются две задачи: бинарная классификация (кошка / собака) и классификация пород. Архитектура сети собственная, обучение с нуля, разбиение на train/test стратифицированное (80/20).

In [ ]:
import os
if not os.path.exists('petfaces'):
    !wget -q http://www.soshnikov.com/permanent/data/petfaces.tar.gz
    !tar xfz petfaces.tar.gz
    !rm petfaces.tar.gz
print(len(os.listdir('petfaces')), 'папок с классами')

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Устройство:', device)

### Загрузка данных

Каждая папка соответствует одной породе. Признак «кошка / собака» берётся из префикса имени папки (`cat_...` / `dog_...`).

In [ ]:
DATA_DIR = 'petfaces'
EXT = ('.jpg', '.jpeg', '.png')

classes = sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)))

paths, labels = [], []
for i, c in enumerate(classes):
    for f in sorted(os.listdir(os.path.join(DATA_DIR, c))):
        if f.lower().endswith(EXT) and not f.startswith('.'):
            paths.append(os.path.join(DATA_DIR, c, f))
            labels.append(i)
labels = np.array(labels)

class_is_cat = np.array([c.lower().startswith('cat') for c in classes])
assert 0 < class_is_cat.sum() < len(classes), 'Не удалось определить кошек/собак по именам папок: ' + str(classes)

counts = np.bincount(labels)
print('Всего изображений:', len(paths))
print('Классов:', len(classes), '| кошек:', class_is_cat.sum(), '| собак:', (~class_is_cat).sum())
print('Изображений на класс: от', counts.min(), 'до', counts.max())
print(classes)

In [ ]:
# Изображения маленькие, поэтому целиком загружаем их в память: обучение идёт заметно быстрее
IMG = 64    # размер входа сети
LOAD = 72   # размер хранения (с запасом для случайных кропов)

images = [Image.open(p).convert('RGB').resize((LOAD, LOAD), Image.BILINEAR) for p in paths]
print('Загружено:', len(images))

### Стратифицированное разбиение

Стратификация по породе автоматически сохраняет и пропорцию кошек и собак.

In [ ]:
idx = np.arange(len(paths))
train_idx, test_idx = train_test_split(idx, test_size=0.2, stratify=labels, random_state=SEED)

split = pd.DataFrame({
    'train': np.bincount(labels[train_idx], minlength=len(classes)),
    'test': np.bincount(labels[test_idx], minlength=len(classes)),
}, index=classes)
split['доля test'] = (split['test'] / (split['train'] + split['test'])).round(3)
print('train:', len(train_idx), '| test:', len(test_idx))
split

In [ ]:
fig, axes = plt.subplots(3, 8, figsize=(16, 6.5))
for ax, i in zip(axes.flat, np.random.choice(train_idx, 24, replace=False)):
    ax.imshow(images[i])
    ax.set_title(classes[labels[i]], fontsize=8)
    ax.axis('off')
plt.show()

### Предобработка и аугментация

Нормализация по среднему и стандартному отклонению обучающей выборки. Для train используются случайные кропы, отражение, небольшой поворот и изменение цвета; для test только изменение размера.

In [ ]:
sample = np.stack([np.asarray(images[i]) for i in train_idx[:1000]]).astype(np.float32) / 255
mean, std = sample.mean(axis=(0, 1, 2)), sample.std(axis=(0, 1, 2))
print('mean:', mean.round(3), 'std:', std.round(3))

train_tf = T.Compose([
    T.RandomResizedCrop(IMG, scale=(0.7, 1.0), ratio=(0.9, 1.1)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ColorJitter(0.2, 0.2, 0.2),
    T.ToTensor(),
    T.Normalize(mean.tolist(), std.tolist()),
])
test_tf = T.Compose([
    T.Resize((IMG, IMG)),
    T.ToTensor(),
    T.Normalize(mean.tolist(), std.tolist()),
])


class PetFaces(Dataset):
    def __init__(self, indices, targets, tf):
        self.indices, self.targets, self.tf = indices, targets, tf

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, k):
        i = self.indices[k]
        return self.tf(images[i]), int(self.targets[i])


def make_loaders(targets, batch_size=64):
    train_dl = DataLoader(PetFaces(train_idx, targets, train_tf), batch_size=batch_size,
                          shuffle=True, num_workers=2, drop_last=True)
    test_dl = DataLoader(PetFaces(test_idx, targets, test_tf), batch_size=256,
                         shuffle=False, num_workers=2)
    return train_dl, test_dl

### Архитектура

Четыре свёрточных блока (два слоя Conv-BatchNorm-ReLU и MaxPool) с числом каналов 32-64-128-256, затем глобальный average pooling, dropout и линейный слой. Одна и та же архитектура используется для обеих задач, различается только число выходов.

In [ ]:
def conv_block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.MaxPool2d(2),
    )


class PetNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(3, 32),     # 64 -> 32
            conv_block(32, 64),    # 32 -> 16
            conv_block(64, 128),   # 16 -> 8
            conv_block(128, 256),  # 8 -> 4
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.4), nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.head(self.features(x))


print('Параметров:', sum(p.numel() for p in PetNet(len(classes)).parameters()))

In [ ]:
@torch.no_grad()
def predict(model, loader):
    model.eval()
    outs, ys = [], []
    for x, y in loader:
        outs.append(model(x.to(device)).cpu())
        ys.append(y)
    return torch.cat(outs), torch.cat(ys)


def train(model, train_dl, test_dl, epochs, lr=3e-3, wd=5e-4):
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(train_dl))
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    hist = {'train_loss': [], 'train_acc': [], 'test_acc': []}

    for ep in range(epochs):
        model.train()
        tl, tc, n = 0.0, 0, 0
        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = crit(out, y)
            opt.zero_grad()
            loss.backward()
            opt.step()
            sched.step()
            tl += loss.item() * len(y)
            tc += (out.argmax(1) == y).sum().item()
            n += len(y)

        # точность на тесте считаем только для наблюдения, модель по ней не отбирается
        logits, ys = predict(model, test_dl)
        acc = (logits.argmax(1) == ys).float().mean().item()
        hist['train_loss'].append(tl / n)
        hist['train_acc'].append(tc / n)
        hist['test_acc'].append(acc)
        print(f'эпоха {ep + 1:2d}/{epochs}  loss {tl / n:.3f}  train acc {tc / n:.3f}  test acc {acc:.3f}')
    return hist


def plot_history(h, title):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    ax[0].plot(h['train_loss'])
    ax[0].set_title(title + ': loss (train)')
    ax[0].set_xlabel('эпоха')
    ax[1].plot(h['train_acc'], label='train')
    ax[1].plot(h['test_acc'], label='test')
    ax[1].set_title(title + ': accuracy')
    ax[1].set_xlabel('эпоха')
    ax[1].legend()
    plt.show()

## Задача 1. Кошка или собака (бинарная классификация)

In [ ]:
bin_labels = (~class_is_cat[labels]).astype(int)   # 0 - кошка, 1 - собака
bin_train_dl, bin_test_dl = make_loaders(bin_labels)

torch.manual_seed(SEED)
bin_model = PetNet(2)
bin_hist = train(bin_model, bin_train_dl, bin_test_dl, epochs=15)
plot_history(bin_hist, 'Кошки / собаки')

In [ ]:
logits, y = predict(bin_model, bin_test_dl)
bin_pred = logits.argmax(1)
bin_acc = (bin_pred == y).float().mean().item()
print(f'Точность бинарного классификатора на тесте: {bin_acc:.4f}')

pd.DataFrame(confusion_matrix(y.numpy(), bin_pred.numpy(), labels=[0, 1]),
             index=['истина: кошка', 'истина: собака'],
             columns=['предсказано: кошка', 'предсказано: собака'])

## Задача 2. Определение породы (мультиклассовая классификация)

In [ ]:
train_dl, test_dl = make_loaders(labels)

torch.manual_seed(SEED)
breed_model = PetNet(len(classes))
breed_hist = train(breed_model, train_dl, test_dl, epochs=40)
plot_history(breed_hist, 'Породы')

### Метрики на тестовой выборке

Точность «кошки против собак» для сети пород считается так: суммируем вероятности всех собачьих классов и сравниваем с 0.5.

In [ ]:
logits, y_true = predict(breed_model, test_dl)
probs = logits.softmax(1)
pred = probs.argmax(1)

acc = (pred == y_true).float().mean().item()
top3 = (probs.topk(3, dim=1).indices == y_true[:, None]).any(1).float().mean().item()

is_dog = torch.tensor(~class_is_cat)
pred_dog = probs[:, is_dog].sum(1) > 0.5
acc_cd = (pred_dog == is_dog[y_true]).float().mean().item()

print(f'Accuracy (top-1) по породам:          {acc:.4f}')
print(f'Top-3 accuracy по породам:            {top3:.4f}')
print(f'Accuracy кошки/собаки (сеть пород):   {acc_cd:.4f}')

In [ ]:
n = len(classes)
cm = confusion_matrix(y_true.numpy(), pred.numpy(), labels=list(range(n)))

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(n))
ax.set_xticklabels(classes, rotation=90, fontsize=7)
ax.set_yticks(range(n))
ax.set_yticklabels(classes, fontsize=7)
ax.set_xlabel('Предсказанный класс')
ax.set_ylabel('Истинный класс')
ax.set_title('Confusion matrix (породы, тестовая выборка)')
for i in range(n):
    for j in range(n):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=6,
                    color='white' if cm[i, j] > cm.max() / 2 else 'black')
fig.colorbar(im, fraction=0.046)
plt.tight_layout()
plt.show()

In [ ]:
# Какие породы сеть путает чаще всего
off = cm.copy()
np.fill_diagonal(off, 0)
for k in np.argsort(off.ravel())[::-1][:10]:
    i, j = divmod(int(k), n)
    print(f'{classes[i]:>30s} -> {classes[j]:<30s} {off[i, j]}')

## Итоги

In [ ]:
pd.DataFrame({
    'метрика': ['Бинарная сеть: accuracy кошки/собаки',
                'Сеть пород: accuracy (top-1)',
                'Сеть пород: top-3 accuracy',
                'Сеть пород: accuracy кошки/собаки'],
    'значение': [round(v, 4) for v in (bin_acc, acc, top3, acc_cd)],
})